# Lesson 2: One tool

**Project:** Agentic Engineering Worklog
**Model:** `qwen3:8b`, running on this PC through Ollama

This lesson gives the model its first **tool**: a way to make something actually happen, instead of only writing text. By the end, a plain-English sentence like "Added vaccine records to herd planner, took about an hour and a half" becomes a real, saved record.

**A note on the saved output:** this notebook was first run in the earlier `Learn-AI-Agent` folder, before the lessons moved into this project. That's why some output shows `Learn-AI-Agent` as a file path and a project name. The code and the results are exactly what ran.

## Where lesson 1 left off

Lesson 1 showed three things this lesson builds on:

1. **A model can only produce text.** When asked to log work, it wrote `LOGGED: Cornerpost Watch | Fixed the alert email bug`. That *looked* like a record, but nothing was saved anywhere.
2. **A model fills gaps with guesses.** It filed "cleaning the meat price data" under a project called "Meat Price Data", a name it made up, because it had no list of your real projects.
3. **The message list is the model's only memory.** Anything the model should know has to be in the list you send.

## The big idea: what is a tool?

A **tool** is an ordinary Python function that you allow the model to ask for. The model never runs it. The model can only reply with a request: "please run `add_entry` with project = Cornerpost Watch and hours = 2." Your code reads that request, runs the function, and sends the result back.

Think of it like an assistant who can't touch the filing cabinet. They can hand you a note saying "please file this under Cornerpost Watch", and you do the filing. You stay in control of what actually happens.

A tool has two parts:

- **The function:** the Python code that does the job, here saving an entry to a file.
- **The description:** a short explanation the model reads, saying what the tool is for and what inputs it needs. The model never sees your code, only this description.

## What we are looking for

By the end of this notebook you should be able to say yes to five things:

1. I can write a tool: an ordinary Python function that does one job.
2. I can describe that tool to the model so it knows when and how to ask for it.
3. I have seen that the model never runs the tool. It asks, and my code runs it.
4. I can send the tool's result back so the model can finish its answer.
5. I can measure how often the model asks for the tool correctly.

This isn't an agent yet. It's fixed steps with one model decision in the middle: should I ask for the tool or not? Lesson 3 adds the loop that turns it into an agent.

**How to run it in PyCharm:** click a code cell and press **Shift + Enter**. Run the cells in order from top to bottom.

## Step 1: Setup

### What we are looking for

A place for the work log to live. The model's requests will turn into lines in this file.

In [19]:
import json
from datetime import datetime
from pathlib import Path

import ollama

MODEL = "qwen3:8b"

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

LOG_FILE = project_root / "data" / "work_log.jsonl"
LOG_FILE.parent.mkdir(exist_ok=True)

print("Log file:", LOG_FILE)

Log file: B:\_Dev\Python\Learn-AI-Agent\data\work_log.jsonl


### The code, line by line

- `import json` loads Python's tool for **JSON**, a common text format for storing data (explained below).
- `from datetime import datetime` loads the tool for working with dates and times, used to stamp each entry.
- `from pathlib import Path` loads a tool for working with file and folder paths.
- `MODEL = "qwen3:8b"` is the same model as lesson 1.
- `project_root = Path.cwd()` gets the folder the notebook is running in. If that's the `notebooks` folder, `.parent` moves up one level to the project folder.
- `LOG_FILE = project_root / "data" / "work_log.jsonl"` builds the full path to the log file. In `pathlib`, the `/` sign joins folder names together.
- `LOG_FILE.parent.mkdir(exist_ok=True)` creates the `data` folder if it doesn't exist. `exist_ok=True` means "don't complain if it's already there".

### How to read it

Your run printed:

```
Log file: B:\_Dev\Python\Learn-AI-Agent\data\work_log.jsonl
```

That's the full path where entries are saved. (It shows `Learn-AI-Agent` because this run happened in the earlier folder.)

*Since this run, the projects moved from `B:\_Dev\Python` to `B:\Development\Python`. The saved output above still shows the old path, because it's a record of what ran. Running the cell again would show the new one.*

### What it means

The log is a **JSON Lines** file, ending in `.jsonl`:

- **JSON** is a way of writing data as text, using labels and values: `{"project": "Herd Planner", "hours": 1.5}`. It looks almost exactly like a Python dictionary.
- **JSON Lines** means one JSON record per line. New entries are simply added to the end of the file.

This format is a good fit for a log: it's easy to add to, easy to read back with Python or pandas, and you can open the file in any text editor to see exactly what the agent saved.

### What's next and why

Before involving the model, Step 2 writes the tool and tests it by hand. If something goes wrong later, you'll know the tool itself works.

## Step 2: Write the tool as plain Python

### What we are looking for

A function that saves one entry, tested by calling it directly, with no model involved.

In [20]:
def add_entry(project, description, hours=None):
    """Save one work log entry to the log file and return a short confirmation."""
    entry = {
        "logged_at": datetime.now().isoformat(timespec="seconds"),
        "project": project,
        "description": description,
        "hours": hours,
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")
    return f"Saved entry for {project}."


print(add_entry("Learn-AI-Agent", "Tested add_entry by hand, no model involved", 0.1))
print()
print(LOG_FILE.read_text(encoding="utf-8"))

Saved entry for Learn-AI-Agent.

{"logged_at": "2026-11-11T19:55:52", "project": "Learn-AI-Agent", "description": "Tested add_entry by hand, no model involved", "hours": 0.1}
{"logged_at": "2026-11-11T19:56:16", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2}
{"logged_at": "2026-11-11T19:56:21", "project": "Herd Planner", "description": "Added vaccine records to herd planner", "hours": 1.5}
{"logged_at": "2026-11-11T19:56:29", "project": "Learn-AI-Agent", "description": "Ran lesson 2 of the agent notebook.", "hours": null}
{"logged_at": "2026-11-12T08:54:46", "project": "Learn-AI-Agent", "description": "Tested add_entry by hand, no model involved", "hours": 0.1}
{"logged_at": "2026-11-12T08:54:49", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2}
{"logged_at": "2026-11-12T08:54:53", "project": "Herd Planner", "description": "Added vaccine records to herd planner", "hours": 1.5}
{"logged_at": "2026-11-12T08:54:

### The code, line by line

- `def add_entry(project, description, hours=None):` defines the tool. It takes three inputs. `hours=None` gives `hours` a **default value**: if no hours are given, it's `None`, Python's word for "nothing".
- `entry = { ... }` builds the record as a dictionary with four labelled values:
    - `logged_at`: the current date and time, written in a standard format like `2026-11-11T19:55:52`.
    - `project`, `description`, `hours`: the three inputs.
- `with open(LOG_FILE, "a", encoding="utf-8") as f:` opens the log file. The `"a"` means **append**: add to the end, never overwrite. The `with` block closes the file automatically when it's done.
- `f.write(json.dumps(entry) + "\n")` turns the dictionary into a line of JSON text and writes it, followed by a line break (`\n`).
- `return f"Saved entry for {project}."` sends back a short confirmation. The model will read this later.
- The last three lines call the tool once by hand, then print the whole log file.

### How to read it

Your run printed `Saved entry for Learn-AI-Agent.`, then **9 lines** of the log file.

Why 9, when the cell saved only one? Because this cell ran several times, on November 11 and 12, and the later cells save entries too. Each run appended its lines. You can see the pattern repeat: the hand test (0.1 hours), Cornerpost Watch (2 hours), Herd Planner (1.5 hours), and "Ran lesson 2" with `"hours": null`.

`null` is how JSON writes Python's `None`. That entry had no hours, and the file records that honestly instead of inventing a number.

### What it means

The tool works before the model is involved at all. That order matters. When an agent misbehaves later, you want to know the tool is sound, so you can look for the problem in the model's choices instead.

Two design choices pay off later:

- **Hours are optional.** An entry is still saved when you don't say how long something took.
- **The function returns a short text result.** The model can only read text, so the result is written for it to read.

## Step 3: Describe the tool to the model

### What we are looking for

The description the model will read. The model can't see your Python code, so this is everything it will ever know about the tool.

In [21]:
ADD_ENTRY_TOOL = {
    "type": "function",
    "function": {
        "name": "add_entry",
        "description": "Save one work log entry. Use this when the user describes work they did.",
        "parameters": {
            "type": "object",
            "properties": {
                "project": {
                    "type": "string",
                    "description": "Name of the project the work was for.",
                },
                "description": {
                    "type": "string",
                    "description": "Short description of what was done.",
                },
                "hours": {
                    "type": "number",
                    "description": "Hours spent. Leave this out if the user did not say.",
                },
            },
            "required": ["project", "description"],
        },
    },
}

TOOLS = [ADD_ENTRY_TOOL]

print(json.dumps(ADD_ENTRY_TOOL, indent=2))

{
  "type": "function",
  "function": {
    "name": "add_entry",
    "description": "Save one work log entry. Use this when the user describes work they did.",
    "parameters": {
      "type": "object",
      "properties": {
        "project": {
          "type": "string",
          "description": "Name of the project the work was for."
        },
        "description": {
          "type": "string",
          "description": "Short description of what was done."
        },
        "hours": {
          "type": "number",
          "description": "Hours spent. Leave this out if the user did not say."
        }
      },
      "required": [
        "project",
        "description"
      ]
    }
  }
}


### The code, line by line

`ADD_ENTRY_TOOL` is a dictionary in a standard shape that Ollama, and most AI services, understand:

- `"type": "function"` says this is a tool the model can request.
- `"name": "add_entry"` is the name the model writes when it wants the tool. It must match a function your code can run.
- `"description"`: when to use it. "Save one work log entry. Use this when the user describes work they did."
- `"parameters"` lists the inputs:
    - each one has a **type**: `string` means text, and `number` means a number.
    - each one has its own short description, telling the model what to put there.
- `"required": ["project", "description"]` lists the inputs the model must always give. `hours` isn't in the list, so it may be left out.
- `TOOLS = [ADD_ENTRY_TOOL]` puts the description in a list, because later lessons offer several tools at once.
- `json.dumps(..., indent=2)` prints it neatly, with indentation.

### How to read it

Your run printed the same description back as indented JSON. Nothing has been sent to the model yet; this just shows what it will receive.

### What it means

Writing good tool descriptions is one of the main skills in building agents. The model decides when to use a tool almost entirely from the description, so:

- **The description sentence** does most of the work. "Use this when the user describes work they did" tells it exactly when to ask.
- **The parameter descriptions** shape the inputs. "Leave this out if the user did not say" is what stops it from inventing hours.

When a model misuses a tool, the first fix is usually better wording here, not more code.

### What's next and why

Now the model gets a real message, with the tool on offer.

## Step 4: Ask the model, with the tool on offer

### What we are looking for

What the model does when it's offered a tool: answer in text, or ask for the tool? Also, does it save anything by itself?

In [22]:
KNOWN_PROJECTS = ["Cornerpost Watch", "Cornerpost Logic", "Herd Planner", "Learn-AI-Agent"]

SYSTEM_PROMPT = (
    "You are a work log assistant. "
    "When the user describes work they did, call the add_entry tool. "
    "Known projects: " + ", ".join(KNOWN_PROJECTS) + ". "
    "Use one of those names when the work matches one. "
    "Only include hours if the user stated them. "
    "If the message is not about work, answer normally without calling a tool."
)


def call_model(messages):
    """One call to the local model with the tools on offer."""
    return ollama.chat(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        think=False,
        options={"temperature": 0},
    )


entries_before = len(LOG_FILE.read_text(encoding="utf-8").splitlines())

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Spent 2 hours fixing the alert email bug in cornerpost watch."},
]

response = call_model(messages)

print("Text reply:    ", repr(response.message.content))
for call in response.message.tool_calls or []:
    print("Tool requested:", call.function.name)
    print("Arguments:     ", dict(call.function.arguments))

entries_after = len(LOG_FILE.read_text(encoding="utf-8").splitlines())
print()
print("Entries in the log file before the call:", entries_before)
print("Entries in the log file after the call: ", entries_after)

Text reply:     ''
Tool requested: add_entry
Arguments:      {'description': 'Fixed the alert email bug', 'hours': 2, 'project': 'Cornerpost Watch'}

Entries in the log file before the call: 9
Entries in the log file after the call:  9


### The code, line by line

- `KNOWN_PROJECTS` is a list of your real project names. This is the fix for lesson 1's made-up "Meat Price Data".
- `SYSTEM_PROMPT` is the standing instructions. Its rules: call `add_entry` for work, use a name from the known list, only include hours if they were stated, and answer normally for anything else. `", ".join(KNOWN_PROJECTS)` turns the list into one line of text with commas between the names.
- `def call_model(messages):` wraps the call in a function, as lesson 1's `ask_model` did. The new part is `tools=TOOLS`, which offers the tool to the model.
- `entries_before` counts the lines in the log file before the call.
- The `messages` list has the system prompt and one user sentence: "Spent 2 hours fixing the alert email bug in cornerpost watch."
- The `print` lines show the model's text reply and any tool it requested, with the inputs it chose.
- `entries_after` counts the lines again, after the call.

`repr(...)` prints text with its quote marks, so an empty reply shows as `''` instead of a blank line.

### How to read it

Your run printed:

```
Text reply:     ''
Tool requested: add_entry
Arguments:      {'description': 'Fixed the alert email bug', 'hours': 2, 'project': 'Cornerpost Watch'}

Entries in the log file before the call: 9
Entries in the log file after the call:  9
```

- **Text reply `''`:** the model wrote no text. It replied *only* with a request.
- **Tool requested `add_entry`:** it picked the right tool.
- **Arguments:** it filled in all three inputs correctly. You typed "cornerpost watch" in lowercase, and it matched that to `Cornerpost Watch` from the known list. It pulled out `2` as a number.
- **9 before, 9 after:** nothing was saved.

### What it means

This is the central idea of the lesson: **the model did not save anything.** It replied with a structured request, "please run `add_entry` with these inputs", and stopped there.

Look at what the model was good at: turning a loose, lowercase sentence into clean, structured data with the right project name and the hours as a number. That's the part ordinary code struggles with, and it's exactly what models are good at.

The known-projects list worked: compare this with lesson 1's invented "Meat Price Data".

### What's next and why

The model asked for something. Nothing happens until your code acts on the request, which is Step 5.

## Step 5: Run the tool yourself

### What we are looking for

Your code carrying out the model's request, and the entry actually appearing in the file.

In [23]:
AVAILABLE_TOOLS = {"add_entry": add_entry}

if not response.message.tool_calls:
    print("The model did not ask for a tool. Run the Step 4 cell again first.")
else:
    call = response.message.tool_calls[0]
    tool_function = AVAILABLE_TOOLS[call.function.name]
    result = tool_function(**call.function.arguments)

    print("Tool result:", result)
    print()
    print("Last line of the log file:")
    print(LOG_FILE.read_text(encoding="utf-8").splitlines()[-1])

Tool result: Saved entry for Cornerpost Watch.

Last line of the log file:
{"logged_at": "2026-11-12T11:17:15", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2}


### The code, line by line

- `AVAILABLE_TOOLS = {"add_entry": add_entry}` is a **lookup table**: tool name on the left, the real Python function on the right.
- `if not response.message.tool_calls:` handles the case where the model didn't ask for anything.
- `call = response.message.tool_calls[0]` takes the first request. (`[0]` means "the first item"; Python counts from 0.)
- `tool_function = AVAILABLE_TOOLS[call.function.name]` looks up the function by the name the model gave.
- `result = tool_function(**call.function.arguments)` runs it with the model's inputs. The `**` **unpacks** the dictionary: `{"project": "X", "hours": 2}` becomes `add_entry(project="X", hours=2)`.
- The last lines print the result and the newest line of the log file. `[-1]` means "the last item".

### How to read it

Your run printed:

```
Tool result: Saved entry for Cornerpost Watch.

Last line of the log file:
{"logged_at": "2026-11-12T11:17:15", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2}
```

Now the entry exists, with exactly the inputs the model chose in Step 4.

### What it means

The work is split cleanly:

- **The model** decided *what* to save, by reading your sentence.
- **Your code** decided *whether* to save it, and did the saving.

`AVAILABLE_TOOLS` is also a **safety boundary**. The model can only cause functions in this table to run. If it asked for a tool called `delete_everything`, there'd be nothing to look up, so nothing would happen. You decide what the model is able to do.

### What's next and why

The entry is saved, but the model doesn't know that yet. It asked, and never heard back. Step 6 tells it.

## Step 6: Send the result back

### What we are looking for

The model's final reply, written *after* it reads what the tool did.

In [24]:
messages.append(response.message)
messages.append({"role": "tool", "tool_name": call.function.name, "content": result})

final = call_model(messages)

print("Final reply:", final.message.content)
print()
print("The message list now holds:")
for m in messages:
    role = m["role"] if isinstance(m, dict) else m.role
    print("  -", role)

Final reply: The entry for fixing the alert email bug in Cornerpost Watch has been saved.

The message list now holds:
  - system
  - user
  - assistant
  - tool


### The code, line by line

- `messages.append(response.message)` adds the model's request (from Step 4) to the message list.
- `messages.append({"role": "tool", ...})` adds the tool's result, with a new role: `tool`. `tool_name` says which tool it came from, and `content` is the text it returned ("Saved entry for Cornerpost Watch.").
- `final = call_model(messages)` calls the model again, with the whole list.
- The last lines print the final reply and the role of each message in the list. (The model's request is a different kind of object from the other messages, so the code checks which kind it is before reading its role.)

### How to read it

Your run printed:

```
Final reply: The entry for fixing the alert email bug in Cornerpost Watch has been saved.

The message list now holds:
  - system
  - user
  - assistant
  - tool
```

The model confirmed the save in plain English, because it read the tool's result.

### What it means

There's a new role, **`tool`**, which carries a tool's result back to the model. The full round trip is:

1. `system`: the standing instructions.
2. `user`: what you typed.
3. `assistant`: the model's *request* to run `add_entry`.
4. `tool`: what `add_entry` returned.
5. The model's final answer, written after reading all four.

This is lesson 1's idea again: **the message list is the memory.** The model only knows the save happened because the result is in the list.

### What's next and why

Offering a tool doesn't mean the model must use it. Step 7 checks that it can tell when *not* to.

## Step 7: The model decides when not to use the tool

### What we are looking for

A plain answer, with no tool request, for a message that isn't about work.

In [25]:
other = call_model([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is the capital of Colorado?"},
])

print("Tool requested:", bool(other.message.tool_calls))
print("Text reply:    ", other.message.content)

Tool requested: False
Text reply:     The capital of Colorado is Denver.


### The code, line by line

The same call as Step 4, with the tool still on offer, but the message is "What is the capital of Colorado?". `bool(...)` turns the tool request list into `True` (there's a request) or `False` (none).

### How to read it

Your run printed:

```
Tool requested: False
Text reply:     The capital of Colorado is Denver.
```

No tool, and a correct plain answer.

### What it means

On every call, the model makes a choice: **answer in text, or ask for a tool.** That one decision is what makes this "level 2" on the ladder (plain code, one model call, agent). An agent makes this same decision over and over, once per step.

The system prompt's last rule, "If the message is not about work, answer normally without calling a tool", is what guides this choice.

### What's next and why

Steps 4 to 6 were done by hand, one cell at a time. Step 8 puts them into one function, so a single sentence goes all the way to a saved record.

## Step 8: Wrap the round trip in one function

### What we are looking for

A complete little program: type a sentence, and get either a saved record and a confirmation, or a plain answer.

In [26]:
def run_one_turn(user_text):
    """Handle one user message: let the model ask for tools once, run them, return its final reply."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    response = call_model(messages)

    if not response.message.tool_calls:
        return response.message.content

    messages.append(response.message)
    for call in response.message.tool_calls:
        name = call.function.name
        arguments = dict(call.function.arguments)
        if name not in AVAILABLE_TOOLS:
            result = f"Error: there is no tool named {name}."
        else:
            try:
                result = AVAILABLE_TOOLS[name](**arguments)
            except TypeError as error:
                result = f"Error: {error}"
        print(f"  [tool] {name}({arguments}) -> {result}")
        messages.append({"role": "tool", "tool_name": name, "content": result})

    final = call_model(messages)
    return final.message.content


for text in [
    "Added vaccine records to herd planner, took about an hour and a half.",
    "Ran lesson 2 of the agent notebook.",
    "How many days are in a leap year?",
]:
    print("You:  ", text)
    print("Model:", run_one_turn(text))
    print()

You:   Added vaccine records to herd planner, took about an hour and a half.
  [tool] add_entry({'description': 'Added vaccine records to herd planner', 'hours': 1.5, 'project': 'Herd Planner'}) -> Saved entry for Herd Planner.
Model: The entry for Herd Planner has been successfully saved. Let me know if you need anything else!

You:   Ran lesson 2 of the agent notebook.
  [tool] add_entry({'description': 'Ran lesson 2 of the agent notebook.', 'project': 'Learn-AI-Agent'}) -> Saved entry for Learn-AI-Agent.
Model: The entry for "Ran lesson 2 of the agent notebook" has been successfully saved under the Learn-AI-Agent project. Let me know if you'd like to log anything else!

You:   How many days are in a leap year?
Model: A leap year has 366 days.



### The code, line by line

`run_one_turn(user_text)` does Steps 4 to 6 in order:

1. Build the message list and call the model.
2. If the model asked for no tool, return its text answer straight away.
3. Otherwise, add the request to the list, then for **each** tool requested:
    - check the name is in `AVAILABLE_TOOLS`. If it isn't, the "result" is an error message.
    - run the function inside `try` / `except`. If the model sent inputs the function can't accept, Python raises a `TypeError`. Instead of crashing, the code catches it and uses the error text as the result.
    - print a `[tool]` line and add the result to the list.
4. Call the model once more and return its final reply.

The loop at the bottom tries three sentences.

### How to read it

Your run printed (shortened):

| You said | Tool call | Final reply |
|---|---|---|
| Added vaccine records to herd planner, took about an hour and a half. | `add_entry`: Herd Planner, **1.5** hours | Saved. |
| Ran lesson 2 of the agent notebook. | `add_entry`: Learn-AI-Agent, **no hours** | Saved under Learn-AI-Agent. |
| How many days are in a leap year? | none | A leap year has 366 days. |

### What it means

All three were handled correctly:

- "an hour and a half" became the number **1.5**. Ordinary code would need special rules for that phrase; the model just understood it.
- For "Ran lesson 2", no time was given, and the model **left hours out instead of inventing a number.** That's the weakness from lesson 1's exercise, and the instruction "Only include hours if the user stated them" prevented it here.
- The leap year question got a plain answer, with no tool.

`run_one_turn` is useful as it stands, but it isn't an agent. Its steps are fixed: ask once, run the tools, ask once more. The model never gets a second chance to ask for something else.

The error handling matters more than it looks. A model will eventually send a bad input. When it does, the program keeps going, and the model reads the error and can explain what went wrong.

### What's next and why

Three tries is encouraging, but it isn't proof. Step 9 measures it properly.

## Step 9: Measure it

### What we are looking for

Scores, not impressions. Six test sentences, each with a known right answer, checked automatically.

This step only looks at what the model *asks for*. It doesn't run the tool, so nothing is added to your log.

In [27]:
TEST_CASES = [
    {"text": "Fixed the alert email bug in Cornerpost Watch for about 2 hours.",
     "tool": True, "project": "Cornerpost Watch", "hours": 2},
    {"text": "Spent an hour on herd planner vaccine records.",
     "tool": True, "project": "Herd Planner", "hours": 1},
    {"text": "Ran lesson 1 of the agent notebook.",
     "tool": True, "project": "Learn-AI-Agent", "hours": None},
    {"text": "Worked on the cornerpost logic company site this morning.",
     "tool": True, "project": "Cornerpost Logic", "hours": None},
    {"text": "What is the capital of Colorado?",
     "tool": False, "project": None, "hours": None},
    {"text": "Remind me what a JSON file is.",
     "tool": False, "project": None, "hours": None},
]


def same_hours(got, expected):
    if expected is None:
        return got is None
    try:
        return float(got) == float(expected)
    except (TypeError, ValueError):
        return False


scores = {"tool": 0, "project": 0, "hours": 0}

for case in TEST_CASES:
    response = call_model([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": case["text"]},
    ])
    calls = response.message.tool_calls or []
    arguments = dict(calls[0].function.arguments) if calls else {}

    tool_ok = bool(calls) == case["tool"]
    project_ok = arguments.get("project") == case["project"]
    hours_ok = same_hours(arguments.get("hours"), case["hours"])

    scores["tool"] += tool_ok
    scores["project"] += project_ok
    scores["hours"] += hours_ok

    print(case["text"])
    print(f"   tool decision: {'right' if tool_ok else 'WRONG'}"
          f"   project: {'right' if project_ok else 'WRONG'} ({arguments.get('project')})"
          f"   hours: {'right' if hours_ok else 'WRONG'} ({arguments.get('hours')})")

total = len(TEST_CASES)
print()
print(f"Tool decision right: {scores['tool']} of {total}")
print(f"Project right:       {scores['project']} of {total}")
print(f"Hours right:         {scores['hours']} of {total}")

Fixed the alert email bug in Cornerpost Watch for about 2 hours.
   tool decision: right   project: right (Cornerpost Watch)   hours: right (2)
Spent an hour on herd planner vaccine records.
   tool decision: right   project: right (Herd Planner)   hours: right (1)
Ran lesson 1 of the agent notebook.
   tool decision: right   project: right (Learn-AI-Agent)   hours: right (None)
Worked on the cornerpost logic company site this morning.
   tool decision: right   project: right (Cornerpost Logic)   hours: right (None)
What is the capital of Colorado?
   tool decision: right   project: right (None)   hours: right (None)
Remind me what a JSON file is.
   tool decision: right   project: right (None)   hours: right (None)

Tool decision right: 6 of 6
Project right:       6 of 6
Hours right:         6 of 6


### The code, line by line

- `TEST_CASES` is a list of six tests. Each one has the sentence, whether a tool *should* be requested, and the expected project and hours. Two sentences aren't about work, so they expect no tool at all.
- `same_hours(got, expected)` compares hours fairly. It treats `2` and `2.0` as equal, and when no hours are expected, the right answer is `None`.
- `scores` starts three counters at 0.
- For each test, the code calls the model, reads its request (if any), and checks three things:
    - **tool decision:** did it ask for a tool exactly when it should?
    - **project:** is the project name exactly right?
    - **hours:** are the hours exactly right, including "none" when none were said?
- `scores[...] += tool_ok` adds 1 for each right answer. In Python, `True` counts as 1 and `False` as 0.

### How to read it

Your run printed one line per test, then:

```
Tool decision right: 6 of 6
Project right:       6 of 6
Hours right:         6 of 6
```

Every check passed, including the two tricky cases:

- "Worked on the cornerpost logic company site this morning." → `Cornerpost Logic`, with **no** hours. "This morning" isn't a number of hours, and the model didn't treat it as one.
- "Remind me what a JSON file is." → no tool. The word "file" didn't fool it into logging.

### What it means

This is an **evaluation**: the same idea as scoring a model on a test set in data science. It replaces "it seemed to work when I tried it" with a number you can compare over time.

Each check also points to a likely fix if it fails:

- **Wrong tool decision:** the tool description or the system prompt's rule about when to log.
- **Wrong project:** the known projects list.
- **Wrong hours:** the model invented a number; firmer wording in the prompt, or a check in code.

Six cases is a tiny test set, so 6 of 6 is encouraging, not proof. One mistake would drop a score to 83%. The habit matters more than the number: every time you change the prompt or a tool description, rerun this cell and see whether the scores move.

## Try it yourself

Change one thing at a time, then rerun Step 9 to see the effect on the scores:

1. In Step 4, edit `KNOWN_PROJECTS` to your real list of projects, rerun Steps 4 to 9, and add a test case for one of the new names.
2. In Step 4, delete the sentence `"Only include hours if the user stated them. "` from the system prompt. Does the hours score drop?
3. In Step 3, change the tool's description to something vague, such as `"A tool."`. Does the tool decision score drop?
4. Add two test cases of your own: one that should be logged and one that shouldn't.

Exercises 2 and 3 show how much the wording, not the code, controls the model's behavior.

## What it all means

- A **tool** is an ordinary Python function plus a description the model can read.
- **The model never runs a tool.** It asks, your code runs it, and your code sends the result back as a `tool` message.
- **The table of available tools limits what the model can cause to happen.** That keeps you in control.
- Giving the model facts (the known project list) stops it from inventing them.
- A small test set turns "does it work?" into scores you can track.

## Words used in this lesson

| Word | Meaning |
|---|---|
| Tool | A Python function the model is allowed to ask for, plus a description of it. |
| Tool call / request | The model's reply saying which tool to run, and with what inputs. |
| Arguments / parameters | The inputs to a function or tool, such as `project` and `hours`. |
| JSON | A text format for data, with labels and values, much like a Python dictionary. |
| JSON Lines (`.jsonl`) | A file with one JSON record per line. |
| Append | Add to the end of a file without changing what's already there. |
| `None` / `null` | "Nothing" in Python and in JSON. Used here for "no hours given". |
| `tool` role | A message carrying a tool's result back to the model. |
| Lookup table | A dictionary that matches names to things, like tool names to functions. |
| Evaluation | Scoring a model on test cases with known right answers. |

## What's next and why

**Lesson 3: The loop.** Right now the model gets one chance to ask for tools, then must answer. That's not enough for requests that need several steps, especially when a later step depends on what an earlier one found.

Lesson 3 adds a second tool, `search_entries`, that reads the log back, and replaces the fixed steps with a **loop**: the model keeps asking for tools until it decides it's finished. That loop is what makes it an agent.